# Generating anticancer peptides with a conditional VAE

DSC4804 Statistical Learning, Group 20.

This notebook runs the whole project from start to finish. Each step is a
script in `src/`, so you can also run them from the terminal.

**The idea.** Anticancer peptides are short chains of amino acids that kill
cancer cells. Finding new ones in a lab is slow and expensive. We train a model
that learns what known anticancer peptides look like, then ask it to write new
ones, and score those with an existing published predictor.

**Run order:** data → train → generate → score → latent analysis.

In [ ]:
import sys, os
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('working in', REPO)

## 1. The data

AntiCP 2.0's "main" dataset: 689 anticancer peptides and 689 antimicrobial
peptides that are *not* anticancer.

The negatives matter. They are not random strings — they are real antimicrobial
peptides. So the model cannot succeed just by learning "this looks like a
peptide"; it has to learn what makes one *anticancer*.

`src/prepare_data.py` cleans the raw files and reports everything it drops.

In [ ]:
!python src/prepare_data.py

In [ ]:
from IPython.display import Image
Image('outputs/length_hist.png')

Anticancer peptides are noticeably shorter than the antimicrobial negatives
(median 18 vs 25). We cap sequences at **25 residues**, following HydrAMP.

Peptides longer than 25 are **dropped, not truncated**. Cutting a peptide short
makes a fragment that is no longer the molecule it came from, and training on
fragments would teach the model to produce fragments.

## 2. The model

A conditional variational autoencoder, adapted from HydrAMP and rewritten small
in PyTorch.

```
peptide x ──► encoder ──► mu, logvar ──► z = mu + sigma·eps ──┐
                                                               ├─► decoder ──► new peptide
label y (1 = anticancer) ──────────────────────────────────────┘
```

**Why the encoder outputs a distribution, not a point.** A plain autoencoder
maps each peptide to a single location. Sample anywhere between two of those
locations and you get noise, because nothing forced the space in between to
mean anything. The VAE instead outputs a mean and a spread, and the KL term
pulls every peptide's distribution toward a standard normal. That makes the
space continuous, which is what lets us sample from it.

**Why `z = mu + sigma·eps`.** Sampling directly from `N(mu, sigma)` would break
backpropagation — you cannot differentiate through a random draw. Writing it
this way puts the randomness in `eps`, which carries no gradient, so `mu` and
`sigma` remain trainable. This is the reparameterisation trick.

**Loss** = reconstruction cross-entropy + β · KL divergence.

**What we dropped from HydrAMP:** the potency/MIC branch, the two internal
classifiers, Gumbel-softmax, and the autoregressive decoder. We have one binary
label and we score with AntiCP 2.0 instead of an internal classifier.

In [ ]:
from model import ConditionalVAE

demo = ConditionalVAE(max_len=25, latent_dim=32, hidden_dim=128)
print(demo)
print(f'\n{sum(p.numel() for p in demo.parameters()):,} parameters')

## 3. Training

Two things to watch:

**Reconstruction accuracy** — the fraction of amino acids put back correctly.
Chance is about 5% (1 in 21).

**Posterior collapse** — the classic VAE failure. If the KL term bites too hard
too early, the encoder gives up and outputs the same distribution for every
peptide. `z` then carries no information and the decoder produces the same
generic sequence every time. The warning signs are KL falling toward zero and
*active units* dropping. The fix is **KL annealing**: hold β at 0 at first so
the model learns to reconstruct, then raise it gradually.

In [ ]:
!python src/train.py --epochs 200 --latent-dim 32 --dropout 0.2

In [ ]:
import json
import matplotlib.pyplot as plt

history = json.load(open('outputs/training_history.json'))
epochs = [h['epoch'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, [h['train_acc'] for h in history], label='train')
axes[0].plot(epochs, [h['val_acc'] for h in history], label='validation')
axes[0].set_title('reconstruction accuracy'); axes[0].legend(frameon=False)

axes[1].plot(epochs, [h['kl'] for h in history], color='#8c3a52')
axes[1].set_title('KL divergence')

axes[2].plot(epochs, [h['active_units'] for h in history], color='#4a6fa5')
axes[2].set_title('active latent units')
axes[2].set_ylim(bottom=0)

for ax in axes: ax.set_xlabel('epoch')
plt.tight_layout(); plt.show()

If the middle panel drops to near zero and the right panel falls toward zero,
the model has collapsed and the generated peptides will all look alike.

## 4. Generating new peptides

Generation is three steps: draw `z` from `N(0, I)`, set `y = 1` for anticancer,
and decode. No encoder and no dataset are needed here — that is the whole point
of making the latent space match a known distribution.

We check four things, and compare against a **composition baseline**: random
peptides built from the same amino-acid frequencies and lengths as the real
anticancer peptides. That baseline knows which letters are common but nothing
about the order they come in. If the cVAE cannot beat it, it has not learned
anything about sequence.

In [ ]:
!python src/generate.py --n 500

## 5. Scoring with AntiCP 2.0

We do **not** train our own classifier. AntiCP 2.0 is published and peer
reviewed, and we use it unchanged.

It needs its own Python 3.7 environment — its model files were pickled with
scikit-learn 0.21 and cannot be loaded by a modern version. See the README.

One caveat worth stating in the report: AntiCP 2.0 was trained on this same
dataset, so it is not an *independent* judge of peptides from this
distribution. It is the standard evaluator in this literature and the right
tool for ranking, but it is not external validation.

In [ ]:
!python src/score_with_anticp.py

In [ ]:
import pandas as pd
ranked = pd.read_csv('outputs/ranked_candidates.csv')
ranked.head(20)

## 6. Inside the latent space

PCA to see the shape of the space, k-means to ask whether the activity split is
something the model found on its own.

- **Adjusted Rand index** — do the clusters match the true labels? 1.0 is
  perfect, 0.0 is chance.
- **Silhouette score** — are the clusters tight and separated, regardless of
  what they mean?

A low ARI is a result, not a failure: it would say activity is not the main
axis of variation in the space, and that length or composition dominates.

In [ ]:
!python src/latent_analysis.py

In [ ]:
Image('outputs/latent_space.png')

## Limitations

Worth saying plainly in the report:

1. **No wet-lab validation.** Everything here is computational. These are
   candidates, not drugs.
2. **The evaluator is not independent.** AntiCP 2.0 was trained on this same
   dataset.
3. **The training set is small** — about 500 anticancer peptides at most once
   the length cap is applied. The train/validation gap in reconstruction shows
   the model memorises more than it generalises.
4. **Near-duplicates exist** between the published train and test splits, so
   held-out reconstruction accuracy reads slightly optimistic. This is why the
   novelty check uses sequence similarity rather than exact matching.